<a href="https://colab.research.google.com/github/zehra44/Project/blob/main/NEXT_WORD_PREDICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [ ]:
data = """Deep learning is a subset of machine learning."""

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])
print(tokenizer.word_index)

{'learning': 1, 'deep': 2, 'is': 3, 'a': 4, 'subset': 5, 'of': 6, 'machine': 7}


In [ ]:
total_words = len(tokenizer.word_index) + 1
print("Total Words:", total_words)

Total Words: 8


In [ ]:
input_sequences = []

for line in data.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [ ]:
print(input_sequences)

[[2, 1], [2, 1, 3], [2, 1, 3, 4], [2, 1, 3, 4, 5], [2, 1, 3, 4, 5, 6], [2, 1, 3, 4, 5, 6, 7], [2, 1, 3, 4, 5, 6, 7, 1]]


In [ ]:
max_len=max([len(x) for x in input_sequences])
input_seq=np.array(pad_sequences(input_sequences,maxlen=max_len,padding='pre'))
print(input_seq)

[[0 0 0 0 0 0 2 1]
 [0 0 0 0 0 2 1 3]
 [0 0 0 0 2 1 3 4]
 [0 0 0 2 1 3 4 5]
 [0 0 2 1 3 4 5 6]
 [0 2 1 3 4 5 6 7]
 [2 1 3 4 5 6 7 1]]


In [ ]:
x = input_seq[:, :-1]
y = input_seq[:, -1]

In [ ]:
x

array([[0, 0, 0, 0, 0, 0, 2],
       [0, 0, 0, 0, 0, 2, 1],
       [0, 0, 0, 0, 2, 1, 3],
       [0, 0, 0, 2, 1, 3, 4],
       [0, 0, 2, 1, 3, 4, 5],
       [0, 2, 1, 3, 4, 5, 6],
       [2, 1, 3, 4, 5, 6, 7]], dtype=int32)

In [ ]:
y

array([1, 3, 4, 5, 6, 7, 1], dtype=int32)

In [ ]:
y = to_categorical(y,num_classes=total_words)

In [ ]:
y

array([[0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0., 0., 0., 0.]])

In [ ]:
x.shape

(7, 7)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=total_words, output_dim=50, input_length=max_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

adam = Adam()

model.compile(loss='categorical_crossentropy',
              optimizer=adam,
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(x,y,epochs=100,verbose=1)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.1429 - loss: 2.0795
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.1429 - loss: 2.0710
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.2857 - loss: 2.0626
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.2857 - loss: 2.0540
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.2857 - loss: 2.0451
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.2857 - loss: 2.0356
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.2857 - loss: 2.0255
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.2857 - loss: 2.0146
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.2857 - loss: 2.0025
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.2857 - loss: 1.9892
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.2857 - loss: 1.9744
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.2857 - los

In [ ]:
def predict_next_word(model, tokenizer, text, max_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    prediction = model.predict(token_list, verbose=0)
    prediction_word_index = np.argmax(prediction)
    predicted_word = tokenizer.index_word[prediction_word_index]
    return predicted_word

In [ ]:
print(predict_next_word(model, tokenizer, "Deep", max_len))

learning


In [ ]:
print(predict_next_word(model, tokenizer, "Deep learning",max_len))

is


In [ ]:
print(predict_next_word(model, tokenizer, "Deep learning is", max_len))

a


In [ ]:
print(predict_next_word(model, tokenizer, "Deep learning is a", max_len))

subset


In [ ]:
print(predict_next_word(model, tokenizer, "Deep learning is a subset", max_len))

of


In [ ]:
print(predict_next_word(model, tokenizer, "Deep learning is a subset of", max_len))

machine


In [ ]:
print(predict_next_word(model, tokenizer, "Deep learning is a subset of machine", max_len))

learning
